# 🧠 MultiModal RAG v3 — End-to-End Colab Ingestion (Baidu Unlimited-OCR)

GPU-accelerated ingestion pipeline for **MultiModal RAG v3**, powered by **Baidu Unlimited-OCR** (`baidu/Unlimited-OCR` — [arXiv:2606.23050](https://arxiv.org/abs/2606.23050)).

### Pipeline Overview:
1. **🚀 Installation & Setup**: GPU dependencies (`transformers`, `qdrant-client`, `sentence-transformers`, `fastembed`, `groq`).
2. **🔑 API Keys & Credentials**: Connect to **Qdrant Cloud** and **Groq API** via environment variables or Colab Secrets.
3. **📄 Baidu Unlimited-OCR PDF Parsing**: One-shot document parsing, HTML table recognition, and formula extraction with a `remove_det_markers` tag cleaner.
4. **🧠 Dense + Sparse Hybrid Embeddings**: **BAAI/bge-small-en-v1.5** (Dense 384d) + **SPLADE** (Sparse Keyword Expansion).
5. **🗄️ Qdrant Cloud Vector Indexing**: Creates a hybrid collection (`multimodal_rag_v3_docs`) and upserts points.
6. **🔍 Multimodal Evaluation Test Suite**: Hybrid RRF search & Groq LLM RAG generation benchmark.

---
### 🛠️ What was fixed in this version (read before running)
This notebook previously failed to run cleanly. The following real issues were found and corrected:

1. **`paddlepaddle-gpu` and `paddleocr` were being installed but never used anywhere in the pipeline.** They're also notoriously fragile, CUDA-version-sensitive pip installs on Colab — almost certainly the single biggest source of install-time failures here. **Removed entirely** (the "PP-Structure" mention in the old title has also been dropped, since it was never actually implemented — this notebook uses Baidu Unlimited-OCR only).
2. **The evaluation cell imported `from openai import OpenAI`, but `openai` was never installed** — guaranteed `ImportError`. The `groq` package *was* already installed but never used. **Switched to the native `groq` client.**
3. **The Groq model `llama-3.3-70b-versatile` is deprecated and scheduled for shutdown on 2026-08-16.** **Switched to `openai/gpt-oss-120b`**, Groq's official recommended replacement.
4. **Two leftover debug cells printed the raw `QDRANT_API_KEY`** — dead code, and a real credential-leak risk if this notebook is ever shared with a saved key in its output. **Removed.**
5. **`torch_dtype=` is deprecated in `transformers==4.57.1`** (this notebook's own previously-saved output already showed that warning). **Switched to `dtype=`.**
6. **The OCR inference call didn't match Baidu Unlimited-OCR's actual documented API** — it was missing required parameters, and the model's own official usage example never captures a return value from `.infer()`, meaning it may write results to disk rather than returning them directly. **Rewritten to match the documented call signature exactly, with a defensive fallback that also checks the output directory** in case the return value is empty. The official tag-stripping post-processor (from the model's own README) is now used instead of the previous ad-hoc regex.


## 🚀 1. Install Dependencies

In [1]:
# Install core packages in Google Colab
# NOTE: paddlepaddle-gpu / paddleocr removed -- they were never imported or used
# anywhere in this notebook, and are a common source of slow/broken installs on
# Colab due to CUDA-version sensitivity. If you actually need PP-Structure table
# recognition, that's a separate, deliberate addition -- don't reintroduce these
# packages just to have them installed.
!pip install -q transformers==4.57.1 pillow matplotlib einops addict easydict pymupdf psutil qdrant-client sentence-transformers fastembed groq httpx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 396.2/396.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.9/323.9 kB 17.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.20.0 requires hugg

## 🔑 2. Configure Credentials & Secrets

In [2]:
import os
from google.colab import userdata

# Set credentials via Colab Secrets or default environment fallback
try:
    QDRANT_URL = userdata.get('QDRANT_URL')
    QDRANT_API_KEY = userdata.get('QDRANT_API_KEY')
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    QDRANT_URL = os.getenv("QDRANT_URL", "https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io")
    QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
    GROQ_API_KEY = os.getenv("GROQ_API_KEY", "gsk_your_groq_api_key_here")

COLLECTION_NAME = "multimodal_rag_v3_docs"

print(f"🌐 Target Qdrant Cloud URL: {QDRANT_URL}")
print(f"📦 Collection Name: {COLLECTION_NAME}")
# NOTE: deliberately not printing QDRANT_API_KEY or GROQ_API_KEY here -- a bare
# `QDRANT_API_KEY` cell used to exist purely to display the raw secret, which is
# a credential-leak risk if this notebook's outputs are ever saved and shared.

🌐 Target Qdrant Cloud URL: https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io
📦 Collection Name: multimodal_rag_v3_docs


## 📄 3. Baidu Unlimited-OCR Engine & PDF Parsing

In [3]:
import re
import os
import uuid
import glob
import tempfile
from pathlib import Path

import fitz  # PyMuPDF
import torch
from transformers import AutoModel, AutoTokenizer

# Official post-processor from the baidu/Unlimited-OCR model card (README), used
# verbatim rather than the simplified regex this notebook used before -- this
# version correctly groups multi-line blocks instead of flattening everything.
DET_RE = re.compile(r"<\|det\|>([^<\s]+)(?:\s*\[[^\]]*\])?\s*<\|/det\|>(.*)", re.DOTALL)

def remove_det_markers(raw: str) -> str:
    """Strip <|det|>type [bbox]<|/det|> markers, group lines belonging to the
    same block with \\n, and separate different blocks with \\n\\n."""
    if not raw:
        return ""
    blocks, cur = [], None
    for line in raw.splitlines():
        line = line.rstrip()
        if not line:
            continue
        m = DET_RE.match(line)
        if m:
            category, content = m.group(1).strip(), m.group(2).strip()
            if category == "image":
                continue
            if cur is not None:
                blocks.append(cur)
            cur = [content] if content else []
            continue
        if cur is None:
            cur = []
        cur.append(line)
    if cur is not None:
        blocks.append(cur)
    return "\n\n".join("\n".join(b) for b in blocks).strip()

print("🤗 Loading Baidu Unlimited-OCR model (baidu/Unlimited-OCR)...")

uocr_model, uocr_tokenizer = None, None
try:
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    uocr_tokenizer = AutoTokenizer.from_pretrained("baidu/Unlimited-OCR", trust_remote_code=True)
    uocr_model = AutoModel.from_pretrained(
        "baidu/Unlimited-OCR",
        trust_remote_code=True,
        use_safetensors=True,
        dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,  # `dtype`, not the deprecated `torch_dtype`
    )
    if torch.cuda.is_available():
        uocr_model = uocr_model.eval().cuda()
    print("✅ Baidu Unlimited-OCR loaded successfully!")
except Exception as exc:
    print(f"⚠️ Baidu Unlimited-OCR load skipped ({exc}). Falling back to plain PyMuPDF text extraction for all pages.")
    uocr_model, uocr_tokenizer = None, None


def _read_infer_output(return_value, output_dir: str) -> str:
    """baidu/Unlimited-OCR's own documented usage example calls .infer(...) as a
    bare statement and never captures a return value -- with save_results=True,
    the model may write results to output_dir instead of returning text directly.
    This checks both: prefer a non-empty return value, else read back the newest
    result file written to output_dir."""
    if isinstance(return_value, str) and return_value.strip():
        return return_value
    candidates = sorted(
        glob.glob(os.path.join(output_dir, "*")),
        key=os.path.getmtime, reverse=True,
    )
    for path in candidates:
        if os.path.isfile(path) and path.lower().endswith((".txt", ".md", ".json", ".mmd")):
            try:
                with open(path, "r", encoding="utf-8") as f:
                    text = f.read()
                if text.strip():
                    return text
            except Exception:
                continue
    return ""


def parse_pdf_to_chunks(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    chunks = []
    print(f"Processing '{pdf_path}' ({len(doc)} pages)...")

    for page_num in range(len(doc)):
        page = doc[page_num]
        text_content = ""

        if uocr_model is not None:
            img_path = f"_temp_uocr_{page_num}.png"
            out_dir = tempfile.mkdtemp(prefix=f"uocr_p{page_num}_")
            try:
                pix = page.get_pixmap(dpi=150)
                pix.save(img_path)
                # Documented single-image call (see model card): "gundam" config
                # is the recommended single-page mode (crop_mode=True).
                raw_output = uocr_model.infer(
                    uocr_tokenizer,
                    prompt="<image>document parsing.",
                    image_file=img_path,
                    output_path=out_dir,
                    base_size=1024, image_size=640, crop_mode=True,
                    max_length=32768,
                    no_repeat_ngram_size=35, ngram_window=128,
                    save_results=True,
                )
                raw_text = _read_infer_output(raw_output, out_dir)
                text_content = remove_det_markers(raw_text)
                if not text_content:
                    print(f"  ⚠️ Page {page_num+1}: OCR returned no usable text (empty return value AND no result file found); "
                          f"falling back to basic PyMuPDF extraction for this page.")
                    text_content = page.get_text("text").strip()
            except Exception as e:
                print(f"  ! Page {page_num+1} OCR failed ({e}), falling back to basic extraction.")
                text_content = page.get_text("text").strip()
            finally:
                if os.path.exists(img_path):
                    os.remove(img_path)
        else:
            text_content = page.get_text("text").strip()

        if text_content:
            modality = "table" if "<table" in text_content.lower() else ("formula" if "\\" in text_content else "text")
            chunks.append({
                "id": str(uuid.uuid4()),
                "text": text_content,
                "page_number": page_num + 1,
                "modality": modality,
                "source_file": Path(pdf_path).name,
            })
    doc.close()
    return chunks

pdf_file = "attention_is_all_you_need.pdf"
if not os.path.exists(pdf_file):
    os.system("wget -q https://arxiv.org/pdf/1706.03762.pdf -O attention_is_all_you_need.pdf")

chunks = parse_pdf_to_chunks(pdf_file)
print(f"\n✅ Extracted {len(chunks)} chunks.")

🤗 Loading Baidu Unlimited-OCR model (baidu/Unlimited-OCR)...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

<|det|>image [198, 208, 818, 761]<|/det|>
<|det|>image_caption [172, 773, 825, 817]<|/det|>Figure 4: Two attention heads, also in layer 5 of 6, apparently involved in anaphora resolution. Top: Full attentions for head 5. Bottom: Isolated attentions from just the word ‘its’ for attention heads 5 and 6. Note that the attentions are very sharp for this word.
<|det|>page_number [492, 936, 508, 948]<|/det|>14
===============save results:===============


other: 100%|██████████| 2/2 [00:00<00:00, 21788.59it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<|det|>image [198, 234, 816, 744]<|/det|>
<|det|>image_caption [172, 758, 825, 802]<|/det|>Figure 5: Many of the attention heads exhibit behaviour that seems related to the structure of the sentence. We give two such examples above, from two different heads from the encoder self-attention at layer 5 of 6. The heads clearly learned to perform different tasks.
<|det|>page_number [492, 936, 508, 948]<|/det|>15
===============save results:===============


other: 100%|██████████| 2/2 [00:00<00:00, 9868.95it/s]


✅ Extracted 15 chunks.


In [7]:
import re
import statistics

VERIFICATION_RESULTS = {}

def _run_single_page_ocr(page_num_0idx: int, doc_path: str = pdf_file):
    """Fresh single-page OCR call, returning BOTH raw tagged output and cleaned text."""
    doc = fitz.open(doc_path)
    page = doc[page_num_0idx]
    img_path = f"_verify_page_{page_num_0idx}.png"
    out_dir = tempfile.mkdtemp(prefix=f"verify_p{page_num_0idx}_")
    pix = page.get_pixmap(dpi=150)
    pix.save(img_path)
    doc.close()
    raw = uocr_model.infer(
        uocr_tokenizer, prompt="<image>document parsing.",
        image_file=img_path, output_path=out_dir,
        base_size=1024, image_size=640, crop_mode=True,
        max_length=32768, no_repeat_ngram_size=35, ngram_window=128,
        save_results=True,
    )
    raw = _read_infer_output(raw, out_dir)
    if os.path.exists(img_path):
        os.remove(img_path)
    return raw, remove_det_markers(raw)


def pdf_to_images(pdf_path, dpi=300, pages=None):
    """From the official model card -- converts PDF pages to image files for infer_multi."""
    doc = fitz.open(pdf_path)
    tmp_dir = tempfile.mkdtemp(prefix="pdf_ocr_")
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    paths = []
    page_indices = pages if pages is not None else range(len(doc))
    for i in page_indices:
        out = os.path.join(tmp_dir, f"page_{i+1:04d}.png")
        doc[i].get_pixmap(matrix=mat).save(out)
        paths.append(out)
    doc.close()
    return paths


print("=" * 70)
print("TEST 1: Sanity check -- known content on page 1")
print("=" * 70)
if uocr_model is None:
    print("⚠️  SKIPPED -- no GPU / model not loaded in this session.")
    VERIFICATION_RESULTS["sanity"] = None
else:
    raw, clean = _run_single_page_ocr(0)
    expected_phrases = ["Attention Is All You Need", "Google"]
    found = [p for p in expected_phrases if p.lower() in clean.lower()]
    passed = len(found) == len(expected_phrases)
    VERIFICATION_RESULTS["sanity"] = passed
    print(f"Expected phrases found: {found} / {expected_phrases}")
    print("✅ PASS" if passed else "❌ FAIL -- model is returning content, but not matching known page-1 text")
    print(f"\nFirst 200 chars of cleaned output:\n{clean[:200]}")


print("=" * 70)
print("TEST 2: Table recognition -- page 8 (Table 2, BLEU scores)")
print("=" * 70)
if uocr_model is None:
    print("⚠️  SKIPPED -- no GPU / model not loaded in this session.")
    VERIFICATION_RESULTS["table"] = None
else:
    raw, clean = _run_single_page_ocr(7)  # 0-indexed -> page 8
    has_table_tag = "<table" in raw.lower() or "<table" in clean.lower()
    has_bleu_numbers = bool(re.search(r"\b2[4-8]\.\d\d?\b", clean))  # BLEU scores in this table are ~24-28
    passed = has_table_tag and has_bleu_numbers
    VERIFICATION_RESULTS["table"] = passed
    print(f"HTML <table> tag present: {has_table_tag}")
    print(f"Plausible BLEU-score numbers present: {has_bleu_numbers}")
    print("✅ PASS" if passed else "❌ FAIL -- table structure is not surviving OCR (may be silently degrading to plain text)")


print("=" * 70)
print("TEST 3: Formula extraction -- page 7 (Equation 3, learning rate formula)")
print("=" * 70)
if uocr_model is None:
    print("⚠️  SKIPPED -- no GPU / model not loaded in this session.")
    VERIFICATION_RESULTS["formula"] = None
else:
    raw, clean = _run_single_page_ocr(6)  # 0-indexed -> page 7
    latex_markers = ["\\cdot", "\\min", "\\text", "warmup", "d_{", "d_ {"]
    found = [m for m in latex_markers if m in clean]
    passed = len(found) >= 2
    VERIFICATION_RESULTS["formula"] = passed
    print(f"LaTeX/equation markers found: {found}")
    print("✅ PASS" if passed else "❌ FAIL -- equation content not being captured with LaTeX-style markup")


print("=" * 70)
print("TEST 4: Long-horizon consistency -- page 1 vs. last page")
print("=" * 70)
print("This is the model's core claimed advantage (constant KV cache via R-SWA).")
print("A normal OCR-LLM without this would typically show degraded/repetitive output")
print("on later pages of a long document. Checking both length and repetitiveness.\n")

if uocr_model is None:
    print("⚠️  SKIPPED -- no GPU / model not loaded in this session.")
    VERIFICATION_RESULTS["long_horizon"] = None
else:
    doc = fitz.open(pdf_file)
    last_page_idx = len(doc) - 1
    doc.close()

    _, clean_first = _run_single_page_ocr(0)
    _, clean_last = _run_single_page_ocr(last_page_idx)

    def repetition_ratio(text):
        words = text.split()
        if not words:
            return 0.0
        return len(set(words)) / len(words)

    wc_first, wc_last = len(clean_first.split()), len(clean_last.split())
    rep_first, rep_last = repetition_ratio(clean_first), repetition_ratio(clean_last)

    length_ok = wc_last > 5
    repetition_ok = rep_last > 0.3 or wc_last < 15

    passed = length_ok and repetition_ok
    VERIFICATION_RESULTS["long_horizon"] = passed
    print(f"Page 1  -- words: {wc_first:4d}, unique-word ratio: {rep_first:.2f}")
    print(f"Page {last_page_idx+1:2d} -- words: {wc_last:4d}, unique-word ratio: {rep_last:.2f}")
    print("✅ PASS" if passed else "❌ FAIL -- last page shows signs of degradation/repetition collapse")
    print(f"\nLast page cleaned output:\n{clean_last[:300]}")


print("=" * 70)
print("TEST 5: infer() vs infer_multi() agreement -- pages 1-3")
print("=" * 70)
if uocr_model is None:
    print("⚠️  SKIPPED -- no GPU / model not loaded in this session.")
    VERIFICATION_RESULTS["multi_agreement"] = None
else:
    single_page_texts = []
    for i in range(3):
        _, clean = _run_single_page_ocr(i)
        single_page_texts.append(clean)

    multi_out_dir = tempfile.mkdtemp(prefix="verify_multi_")
    image_paths = pdf_to_images(pdf_file, dpi=300, pages=range(3))
    raw_multi = uocr_model.infer_multi(
        uocr_tokenizer, prompt="<image>Multi page parsing.",
        image_files=image_paths, output_path=multi_out_dir,
        image_size=1024, max_length=32768,
        no_repeat_ngram_size=35, ngram_window=1024, save_results=True,
    )
    raw_multi = _read_infer_output(raw_multi, multi_out_dir)
    clean_multi = remove_det_markers(raw_multi)

    def key_phrases(text, n=3, min_len=8):
        words = text.split()
        return [" ".join(words[i:i+4]) for i in range(0, max(len(words)-4, 0), max(len(words)//10, 1))][:n]

    total_checked, total_found = 0, 0
    for page_i, text in enumerate(single_page_texts):
        for phrase in key_phrases(text):
            if len(phrase) < 8:
                continue
            total_checked += 1
            if phrase.lower() in clean_multi.lower():
                total_found += 1

    agreement_ratio = total_found / total_checked if total_checked else 0
    passed = agreement_ratio > 0.5
    VERIFICATION_RESULTS["multi_agreement"] = passed
    print(f"Phrase agreement between infer() and infer_multi(): {total_found}/{total_checked} ({agreement_ratio:.0%})")
    print("✅ PASS" if passed else "❌ FAIL -- per-page infer() and documented infer_multi() are diverging; "
          "investigate before trusting the main pipeline's per-page approach")


print("=" * 70)
print("SUMMARY")
print("=" * 70)
labels = {
    "sanity": "1. Sanity check",
    "table": "2. Table recognition",
    "formula": "3. Formula extraction",
    "long_horizon": "4. Long-horizon consistency",
    "multi_agreement": "5. infer() vs infer_multi() agreement",
}
for key, label in labels.items():
    result = VERIFICATION_RESULTS.get(key)
    status = "⚠️  SKIPPED (no GPU)" if result is None else ("✅ PASS" if result else "❌ FAIL")
    print(f"{label:45s} {status}")

print("""
Not automatable in this cell -- check manually:
  6. Multi-column reading order  -> run on a 2-column paper (e.g. arXiv 1512.03385)
                                     and read the output start-to-finish: does it follow
                                     column 1 top-to-bottom then column 2, or does it
                                     jump back and forth mid-sentence?
  7. Non-English / multilingual  -> run on a Chinese-language PDF and check the output
                                     is coherent Chinese, not garbled or empty.
""")

TEST 1: Sanity check -- known content on page 1


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<|det|>aside_text [23, 268, 61, 702]<|/det|>arXiv:1706.03762v7 [cs.CL] 2 Aug 2023
<|det|>text [200, 90, 800, 143]<|/det|>Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
<|det|>title [343, 186, 654, 209]<|/det|>Attention Is All You Need
<|det|>text [189, 294, 355, 338]<|/det|>Ashish Vaswani*  
Google Brain  
avaswani@google.com
<|det|>text [374, 294, 508, 338]<|/det|>Noam Shazeer*  
Google Brain  
noam@google.com
<|det|>text [526, 294, 668, 338]<|/det|>Niki Parmar*  
Google Research  
nikip@google.com
<|det|>text [687, 294, 813, 338]<|/det|>Jakob Uszkoreit*  
Google Research  
usz@google.com
<|det|>text [204, 358, 346, 400]<|/det|>Llion Jones*  
Google Research  
llion@google.com
<|det|>text [382, 357, 558, 400]<|/det|>Aidan N. Gomez \( ^{*} \)  \( ^{\dagger} \) 
University of Toronto
aidan@cs.toronto.edu
<|det|>text [593, 357, 795, 400]<|/det|>Łukasz Kaiser*  
G

image: 0it [00:00, ?it/s]
other: 100%|██████████| 17/17 [00:00<00:00, 61787.84it/s]

Expected phrases found: ['Attention Is All You Need', 'Google'] / ['Attention Is All You Need', 'Google']
✅ PASS

First 200 chars of cleaned output:
arXiv:1706.03762v7 [cs.CL] 2 Aug 2023
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or schol
TEST 2: Table recognition -- page 8 (Table 2, BLEU scores)


...

<|det|>image [319, 89, 681, 499]<|/det|>
<|det|>image_caption [338, 509, 658, 525]<|/det|>Figure 1: The Transformer - model architecture.
<|det|>text [171, 549, 828, 593]<|/det|>The Transformer follows this overall architecture using stacked self-attention and point-wise, fully connected layers for both the encoder and decoder, shown in the left and right halves of Figure 1, respectively.
<|det|>title [172, 608, 416, 623]<|/det|>3.1 Encoder and Decoder Stacks
<|det|>text [171, 633, 828, 732]<|/det|>Encoder: The encoder is composed of a stack of N = 6 identical layers. Each layer has two sub-layers. The first is a multi-head self-attention mechanism, and the second is a simple, position-wise fully connected feed-forward network. We employ a residual connection [11] around each of the two sub-layers, followed by layer normalization [1]. That is, the output of each sub-layer is LayerNorm( \( x + \text{Sublayer}(x) \) ), where  \( \text{Sublayer}(x) \)  is the function implemented by the s

other: 100%|██████████| 8/8 [00:00<00:00, 41579.22it/s]
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


<PAGE><|det|>aside_text [23, 268, 61, 702]<|/det|>arXiv:1706.03762v7 [cs.CL] 2 Aug 2023
<|det|>text [198, 90, 800, 144]<|/det|>Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works.
<|det|>title [342, 185, 655, 209]<|/det|>Attention Is All You Need
<|det|>text [214, 294, 332, 308]<|/det|>Ashish Vaswani*
<|det|>text [222, 310, 317, 323]<|/det|>Google Brain
<|det|>text [188, 325, 353, 338]<|/det|>avaswani@google.com
<|det|>text [386, 294, 500, 308]<|/det|>Noam Shazeer*
<|det|>text [392, 310, 486, 323]<|/det|>Google Brain
<|det|>text [373, 325, 506, 338]<|/det|>noam@google.com
<|det|>text [549, 294, 650, 308]<|/det|>Niki Parmar*
<|det|>text [536, 310, 655, 323]<|/det|>Google Research
<|det|>text [521, 325, 670, 338]<|/det|>nikip@google.com
<|det|>text [691, 294, 813, 308]<|/det|>Jakob Uszkoreit*
<|det|>text [692, 310, 808, 323]<|/det|>Google Research
<|det|>text [686, 325

image_page_0: 0it [00:00, ?it/s]
other_page_0: 100%|██████████| 32/32 [00:00<00:00, 71659.22it/s]
image_page_1: 0it [00:00, ?it/s]
other_page_2: 100%|██████████| 8/8 [00:00<00:00, 51941.85it/s]

Phrase agreement between infer() and infer_multi(): 5/9 (56%)
✅ PASS
SUMMARY
1. Sanity check                               ✅ PASS
2. Table recognition                          ✅ PASS
3. Formula extraction                         ✅ PASS
4. Long-horizon consistency                   ✅ PASS
5. infer() vs infer_multi() agreement         ✅ PASS

Not automatable in this cell -- check manually:
  6. Multi-column reading order  -> run on a 2-column paper (e.g. arXiv 1512.03385)
                                     and read the output start-to-finish: does it follow
                                     column 1 top-to-bottom then column 2, or does it
                                     jump back and forth mid-sentence?
  7. Non-English / multilingual  -> run on a Chinese-language PDF and check the output
                                     is coherent Chinese, not garbled or empty.



## 🧠 4. Generate Dense + Sparse Hybrid Embeddings

In [4]:
from sentence_transformers import SentenceTransformer
from fastembed import SparseTextEmbedding

print("Loading embedding models...")
dense_model = SentenceTransformer("BAAI/bge-small-en-v1.5")
sparse_model = SparseTextEmbedding(model_name="prithivida/Splade_PP_en_v1")

texts = [c["text"] for c in chunks]
print(f"Generating dense embeddings (BAAI/bge-small-en-v1.5) for {len(texts)} chunks...")
dense_embeddings = dense_model.encode(texts, show_progress_bar=True, normalize_embeddings=True)

print("Generating sparse embeddings (SPLADE)...")
sparse_embeddings = list(sparse_model.embed(texts))

print(f"✅ Dense & Sparse embeddings generated successfully for {len(chunks)} chunks.")

Loading embedding models...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

Generating dense embeddings (BAAI/bge-small-en-v1.5) for 15 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generating sparse embeddings (SPLADE)...
✅ Dense & Sparse embeddings generated successfully for 15 chunks.


## 🗄️ 5. Index Vectors in Persistent Qdrant Cloud

In [5]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, SparseVectorParams, Distance, PointStruct, SparseVector

# Connect to Persistent Qdrant Cloud or fallback to local/memory
if QDRANT_URL and QDRANT_URL.startswith("http"):
    print(f"🌐 Connecting to Qdrant Cluster at: {QDRANT_URL}")
    client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY or None)
else:
    print("ℹ️ Using in-memory Qdrant instance (:memory:).")
    client = QdrantClient(":memory:")

# Recreate collection with Hybrid (Dense + Sparse) vector configuration
if client.collection_exists(COLLECTION_NAME):
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={"dense": VectorParams(size=384, distance=Distance.COSINE)},
    sparse_vectors_config={"sparse": SparseVectorParams()},
)

# Upsert points
points = []
for i, chunk in enumerate(chunks):
    s_emb = sparse_embeddings[i]
    points.append(PointStruct(
        id=chunk["id"],
        vector={
            "dense": dense_embeddings[i].tolist(),
            "sparse": SparseVector(indices=s_emb.indices.tolist(), values=s_emb.values.tolist()),
        },
        payload={
            "text": chunk["text"],
            "page_number": chunk["page_number"],
            "modality": chunk["modality"],
            "source_file": chunk["source_file"],
        },
    ))

client.upsert(collection_name=COLLECTION_NAME, points=points)
print(f"✅ Successfully indexed {len(points)} chunks into Qdrant collection '{COLLECTION_NAME}'.")

🌐 Connecting to Qdrant Cluster at: https://491a2582-7d00-4644-9809-a3faab7fab8a.eu-west-2-0.aws.cloud.qdrant.io
✅ Successfully indexed 15 chunks into Qdrant collection 'multimodal_rag_v3_docs'.


## 🔍 6. Multimodal RAG Hybrid Evaluation & Benchmark

In [6]:
from qdrant_client.models import Prefetch, FusionQuery, Fusion
from groq import Groq  # native Groq SDK -- was previously imported as `openai.OpenAI`,
                        # which was never installed and would raise ImportError.

def hybrid_search(query: str, top_k: int = 3):
    q_dense = dense_model.encode(query, normalize_embeddings=True).tolist()
    q_sparse_obj = list(sparse_model.embed([query]))[0]
    q_sparse = SparseVector(indices=q_sparse_obj.indices.tolist(), values=q_sparse_obj.values.tolist())

    res = client.query_points(
        collection_name=COLLECTION_NAME,
        prefetch=[
            Prefetch(query=q_dense, using="dense", limit=top_k * 2),
            Prefetch(query=q_sparse, using="sparse", limit=top_k * 2),
        ],
        query=FusionQuery(fusion=Fusion.RRF),
        limit=top_k,
        with_payload=True,
    )
    return res.points

TEST_QUERIES = [
    "What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks?",
    "What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)?",
    "Describe the visual architecture of the Transformer model from Figure 1, detailing the Encoder and Decoder sub-layers.",
    "Why is Scaled Dot-Product Attention divided by sqrt(d_k) when d_k is large?",
]

# NOTE: llama-3.3-70b-versatile is deprecated on Groq (shutdown 2026-08-16).
# openai/gpt-oss-120b is Groq's official recommended replacement.
GROQ_MODEL = "openai/gpt-oss-120b"

groq_client = None
if GROQ_API_KEY and not GROQ_API_KEY.startswith("gsk_your_"):
    groq_client = Groq(api_key=GROQ_API_KEY)

for idx, query in enumerate(TEST_QUERIES, 1):
    print(f"\n--- [Query {idx}] {query} ---")
    results = hybrid_search(query, top_k=3)
    passages = []
    for r in results:
        p_no = r.payload.get("page_number", "?")
        txt = r.payload.get("text", "")
        passages.append(f"(Page {p_no}) {txt}")
        print(f" • [Page {p_no}] Score: {r.score:.4f} | Preview: {txt[:120]}...")

    if groq_client:
        resp = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": "You are a scientific assistant. Answer concisely using provided context."},
                {"role": "user", "content": "Context:\n" + "\n\n".join(passages) + f"\n\nQuestion: {query}"},
            ],
        )
        print("\n🤖 Groq Answer:")
        print(resp.choices[0].message.content)
    else:
        print("\n⚠️ GROQ_API_KEY not set -- skipping LLM answer generation for this query.")


--- [Query 1] What are the BLEU scores for Transformer (big) on the WMT 2014 English-to-German and English-to-French translation tasks? ---
 • [Page 8] Score: 1.0000 | Preview: Table 2: The Transformer achieves better BLEU scores than previous state-of-the-art models on the English-to-German and ...
 • [Page 9] Score: 0.6667 | Preview: Table 3: Variations on the Transformer architecture. Unlisted values are identical to those of the base model. All metri...
 • [Page 10] Score: 0.4500 | Preview: Table 4: The Transformer generalizes well to English constituency parsing (Results are on Section 23 of WSJ)
<table><tr>...

🤖 Groq Answer:
- **English → German (WMT 2014):** 28.4 BLEU  
- **English → French (WMT 2014):** 41.8 BLEU

--- [Query 2] What is the mathematical formula for Scaled Dot-Product Attention, including the scaling factor sqrt(d_k)? ---
 • [Page 4] Score: 1.0000 | Preview: ![](images/0.jpg)
![](images/1.jpg)
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head At